<a href="https://colab.research.google.com/github/AnkaRezaa/machine_learning_254107023002/blob/main/jobsheet%20dataset%20AntiBurnOut/JS14/11_Deep_Learning_1D_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JS14: Deep Learning 1D-CNN untuk Data Tabular & Teks - Dataset AntiBurnOut

**Identitas Mahasiswa:**
- **Nama**  : MUHAMMAD ANKA REZA
- **NIM**   : 254107023002
- **Kelas** : TI/3H
- **Dataset**: `mental_health_burnout_prediction_dataset.csv`

---

### Adaptasi Modul (Dari 2D-CNN Citra ke 1D-CNN Tabular/Sekensial):
Pada praktikum kurikulum standar, JS14 berfokus pada **Convolutional Neural Network (CNN)** 2 dimensi untuk klasifikasi citra. Namun, sesuai arahan praktikum kustom AntiBurnOut:
- **Batasan Praktikum**: Tidak menggunakan file gambar/citra; seluruh pipeline berbasis pemrosesan teks dan tabular.
- **Solusi Metodologis**: Menerapkan arsitektur **1D-CNN (`Conv1D`)**.

### Mengapa 1D-CNN Sangat Relevan untuk Data Tabular & Teks?
1. **Pendeteksian Fitur Lokal (Local Feature Extraction):**
   Lapisan konvolusi 1D menggeser filter berukuran $k$ (kernel size) melintasi urutan variabel input:
   $$y_t = f\left( \sum_{i=0}^{k-1} w_i \cdot x_{t+i} + b \right)$$
   Filter ini mampu menangkap korelasi gabungan antar-variabel bertetangga (misal: kombinasi jam kerja tinggi + jam tidur rendah + tingkat kecemasan).
2. **Invariant Transformasi & Bobot Terbagi (*Weight Sharing*):**
   Parameter filter dibagikan di seluruh dimensi fitur sehingga mengurangi risiko *overfitting* dibanding model dense berukuran raksasa.
3. **Kecepatan Inferensi:**
   Arsitektur 1D-CNN membutuhkan komputasi matriks yang sangat ringan, menjadikannya sangat efisien untuk sistem skrining kesehatan mental skala besar.

In [ ]:
# 1. Import Library Kunci
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

np.random.seed(42)
tf.random.set_seed(42)
print(f"TensorFlow: {tf.__version__}")

In [ ]:
# 2. Memuat Dataset AntiBurnOut
possible_paths = [
    '../mental_health_burnout_prediction_dataset.csv',
    'mental_health_burnout_prediction_dataset.csv',
    '../../mental_health_burnout_prediction_dataset.csv'
]
csv_path = None
for p in possible_paths:
    if os.path.exists(p):
        csv_path = p
        break

if csv_path is None:
    raise FileNotFoundError("Dataset mental_health_burnout_prediction_dataset.csv tidak ditemukan!")

df = pd.read_csv(csv_path)
print(f"Dataset loaded: {df.shape}")

In [ ]:
# 3. Rekayasa Fitur & Transformasi Dimensi Tensor untuk 1D-CNN
# Kelompokkan fitur dalam urutan semantik yang berdekatan:
# [Pekerjaan] -> [Kondisi Mental] -> [Gaya Hidup & Usia]
features_ordered = [
    # Blok 1: Beban Kerja & Kepuasan
    'Work_Hours_Per_Week', 'Job_Satisfaction', 'Work_Life_Balance',
    # Blok 2: Indikator Psikologis
    'Anxiety_Score', 'Depression_Score', 'Mood_Score',
    # Blok 3: Indikator Fisik & Gaya Hidup
    'Sleep_Hours', 'Physical_Activity_Hours', 'Screen_Time_Hours', 'Age'
]

target_col = 'Burnout_Risk'
data = df[features_ordered + [target_col]].copy()

# Imputasi nilai hilang
for col in features_ordered:
    data[col] = data[col].fillna(data[col].median())
data[target_col] = data[target_col].fillna(data[target_col].mode()[0])

# Map target klasifikasi: Low: 0, Moderate: 1, High: 2
risk_mapping = {'Low': 0, 'Moderate': 1, 'High': 2}
y = data[target_col].map(risk_mapping).values
X = data[features_ordered].values

# Split data 70% Train, 15% Val, 15% Test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

# Standardisasi
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Reshape ke bentuk tensor 3D: (samples, time_steps/features, channels)
# Di sini shape menjadi: (N, 10, 1)
X_train_cnn = np.expand_dims(X_train_scaled, axis=2)
X_val_cnn = np.expand_dims(X_val_scaled, axis=2)
X_test_cnn = np.expand_dims(X_test_scaled, axis=2)

print(f"Tensor Shape Input 1D-CNN: {X_train_cnn.shape} (batch, features, channels)")

In [ ]:
# 4. Membangun Arsitektur 1D-CNN dengan Keras
cnn_model = keras.Sequential([
    # Input Layer dengan dimensi (10, 1)
    layers.Input(shape=(X_train_cnn.shape[1], 1)),
    
    # Blok Konvolusi 1: Ekstraksi fitur lokal antar-variabel bertetangga
    layers.Conv1D(filters=32, kernel_size=3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),
    
    # Blok Konvolusi 2: Ekstraksi fitur abstrak tingkat lanjut
    layers.Conv1D(filters=64, kernel_size=3, padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling1D(),
    
    # Fully Connected Dense Layers
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    
    # Output Softmax (3 Kelas Risiko Burnout)
    layers.Dense(3, activation='softmax')
], name="AntiBurnOut_1D_CNN")

cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

In [ ]:
# 5. Pelatihan Model 1D-CNN
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

history_cnn = cnn_model.fit(
    X_train_cnn, y_train,
    validation_data=(X_val_cnn, y_val),
    epochs=25,
    batch_size=64,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
# 6. Visualisasi Kurva Pelatihan & Evaluasi Model 1D-CNN
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss Plot
axes[0].plot(history_cnn.history['loss'], label='Train Loss', color='#2563eb', linewidth=2)
axes[0].plot(history_cnn.history['val_loss'], label='Val Loss', color='#dc2626', linestyle='--', linewidth=2)
axes[0].set_title('Kurva Loss Pelatihan 1D-CNN', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Sparse Crossentropy Loss')
axes[0].grid(True, linestyle=':', alpha=0.6)
axes[0].legend()

# Accuracy Plot
axes[1].plot(history_cnn.history['accuracy'], label='Train Accuracy', color='#16a34a', linewidth=2)
axes[1].plot(history_cnn.history['val_accuracy'], label='Val Accuracy', color='#f59e0b', linestyle='--', linewidth=2)
axes[1].set_title('Kurva Akurasi Pelatihan 1D-CNN', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Akurasi')
axes[1].grid(True, linestyle=':', alpha=0.6)
axes[1].legend()
plt.tight_layout()
plt.show()

# Prediksi pada Test Data
y_pred_probs = cnn_model.predict(X_test_cnn)
y_pred = np.argmax(y_pred_probs, axis=1)

target_names = ['Low', 'Moderate', 'High']
print("=== Laporan Klasifikasi Model 1D-CNN AntiBurnOut ===")
print(classification_report(y_test, y_pred, target_names=target_names))

# Heatmap Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=target_names, yticklabels=target_names)
plt.title('Confusion Matrix - 1D-CNN', fontsize=12, fontweight='bold')
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.show()

### Perbandingan Arsitektur: 1D-CNN vs MLP Standar (JS13):
1. **Parameter Efficiency:** 1D-CNN memanfaatkan konvolusi berbobot terbagi (*weight sharing*) dan *pooling*, menghasilkan parameter yang lebih efisien dan terhindar dari overfitting.
2. **Local Feature Interaction:** Filter 1D secara simultan mengkombinasikan atribut yang memiliki kedekatan domain (misalnya stres dan kecemasan) secara lebih alami.
3. **Skalabilitas:** Model 1D-CNN sangat adaptif jika di masa mendatang terdapat fitur temporal/time-series (seperti log harian denyut jantung atau durasi tidur berkala).